# 최종


In [ ]:
"""
MIMIC-IV to HL7 FHIR Conversion Multi-Agent Pipeline (Advanced Dual-RAG)

아키텍처 구성:
1. DatabaseGateway: 화이트리스트 기반의 보안 DB 접근 (SELECT 전용 샌드박스).
2. FHIRSchemaKnowledgeBuilder: FHIR R4 표준 구조 지식 구축을 위한 RAG.
3. MedicalKnowledgeRetriever: 임상 용어 및 매핑 로직 지식을 위한 RAG.
4. Multi-Agent Pipeline: 
    - Manager: 전체 작업 조율 및 데이터 추출 담당.
    - Mapper: CoT(Chain of Thought)를 통한 FHIR JSON 생성.
    - Critic: 생성 결과물 검증 및 자가 수정을 위한 피드백 제공.
"""

import os
import json
import ast
import re
from typing import Dict, List, Tuple, Optional, Any

import pandas as pd
from pydantic import BaseModel, Field
from sqlalchemy import create_engine, text, inspect
from langchain_community.utilities import SQLDatabase
from langchain_core.documents import Document
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from langchain_community.tools import TavilySearchResults
from datetime import datetime
from config import config

# =============================================================================
# 1. Database Gateway (보안 샌드박스)
# =============================================================================

class DatabaseGateway:
    """에이전트의 데이터베이스 탐색 범위를 제한하는 샌드박스 게이트웨이"""
    
    def __init__(self, uri: str, schemas: List[str], allowed_tables: List[str]):
        self.engine = create_engine(uri, connect_args={"options": f"-c search_path={','.join(schemas)}"})
        self.schemas = schemas
        self.allowed_tables = [table.lower() for table in allowed_tables]
        # 화이트리스트에 등록된 테이블만 접근 가능하도록 SQLDatabase 객체 제한
        self.db = SQLDatabase(self.engine, include_tables=self.allowed_tables)

    def _extract_tables_from_query(self, query: str) -> List[str]:
        """SQL 문자열에서 테이블 이름을 추출하여 화이트리스트 포함 여부 확인"""
        pattern = re.compile(r'\b(?:FROM|JOIN)\s+([a-zA-Z0-9_]+)\b', re.IGNORECASE)
        matches = pattern.findall(query)
        return [match.lower() for match in matches]
        
    def execute_safe_query(self, query: str) -> List[Any]:
        """보안 검사(SELECT 전용, 화이트리스트 테이블) 후 쿼리 실행 (딕셔너리 반환)"""
        if not query.strip().upper().startswith("SELECT"):
            raise ValueError("[Security Block] 오직 SELECT 쿼리만 허용됩니다.")
        
        queried_tables = self._extract_tables_from_query(query)
        if not queried_tables:
            raise ValueError("[Security Block] 쿼리에서 대상 테이블을 식별할 수 없습니다.")
            
        for table in queried_tables:
            if table not in self.allowed_tables:
                raise ValueError(f"[Security Block] 접근 거부: '{table}' 테이블은 허용 목록에 없습니다.")

        try:
            # 수정된 부분: SQLAlchemy 연결을 통해 결과를 딕셔너리로 안전하게 가져옵니다.
            with self.engine.connect() as conn:
                result = conn.execute(text(query))
                # 로우(row) 데이터를 컬럼명이 키(key)인 딕셔너리 형태로 변환하여 리스트로 반환
                return [dict(row._mapping) for row in result]
        except Exception as e:
            return [{"error": f"Query Execution Failed: {str(e)}"}]

    def get_schema_metadata(self) -> Dict[str, Any]:
        """허용된 테이블에 대한 메타데이터만 추출"""
        inspector = inspect(self.engine)
        metadata = {}
        for schema in self.schemas:
            metadata[schema] = {}
            for table in self.allowed_tables:
                if inspector.has_table(table, schema=schema):
                    columns = inspector.get_columns(table, schema=schema)
                    metadata[schema][table] = [
                        {"column_name": col["name"], "data_type": str(col["type"])} 
                        for col in columns
                    ]
        return metadata

# =============================================================================
# 2. RAG 시스템: FHIR Schema Builder & Medical Knowledge Retriever
# =============================================================================

class FHIRSchemaKnowledgeBuilder:
    """fhir.schema.json을 파싱하여 구조적 지식 벡터 DB 구축"""
    
    def __init__(self, persist_directory: str = "./fhir_schema_vectordb"):
        self.persist_directory = persist_directory
        self.embedding_model = HuggingFaceEmbeddings(model_name="microsoft/harrier-oss-v1-270m")
        self.text_splitter = RecursiveCharacterTextSplitter(chunk_size=800, chunk_overlap=100)

    def _parse_fhir_schema(self, json_data: dict) -> List[Document]:
        docs = []
        definitions = json_data.get("definitions", {})
        for def_name, def_data in definitions.items():
            # 리소스 개요
            desc = def_data.get("description", "No description available.")
            overview_text = f"Resource/DataType: {def_name}\nDescription: {desc}"
            docs.append(Document(page_content=overview_text, metadata={"category": "overview", "definition": def_name}))
            
            # 속성 상세 내역
            properties = def_data.get("properties", {})
            for prop_name, prop_data in properties.items():
                prop_desc = prop_data.get("description", "")
                type_info = prop_data.get("type", "")
                
                if "$ref" in prop_data:
                    type_info += f" (Ref: {prop_data['$ref'].split('/')[-1]})"
                elif "items" in prop_data and "$ref" in prop_data["items"]:
                    type_info += f" Array of {prop_data['items']['$ref'].split('/')[-1]}"
                
                enum_info = f"\nEnum: {', '.join(prop_data['enum'])}" if "enum" in prop_data else ""
                pattern_info = f"\nPattern: {prop_data['pattern']}" if "pattern" in prop_data else ""

                prop_text = (f"Resource: {def_name}\nProperty: {prop_name}\nType: {type_info}\n"
                             f"Description: {prop_desc}{enum_info}{pattern_info}")
                
                docs.append(Document(page_content=prop_text, metadata={"category": "property", "definition": def_name, "property": prop_name}))
        return docs

    def get_or_build_db(self, schema_file_path: str = "./db/fhir.schema.json") -> Chroma:
        if os.path.exists(self.persist_directory):
            print(f"[*] 기존 FHIR Schema Vector DB 로딩 중: {self.persist_directory}")
            return Chroma(persist_directory=self.persist_directory, embedding_function=self.embedding_model)
        
        print(f"[*] FHIR Schema Vector DB 구축 중 ({schema_file_path})...")
        with open(schema_file_path, 'r', encoding='utf-8') as f:
            data = json.load(f)
        
        all_docs = self._parse_fhir_schema(data)
        split_docs = self.text_splitter.split_documents(all_docs)
        
        return Chroma.from_documents(
            documents=split_docs, 
            embedding=self.embedding_model, 
            persist_directory=self.persist_directory
        )

import os
from typing import List, Any
from langchain_core.documents import Document
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

class MedicalKnowledgeRetriever:
    """임상 용어(MIMIC Local Codes) 및 변환 정책 가이드라인 빌더/리트리버"""
    
    def __init__(self, persist_directory: str = "./mimic_docs_vectordb"):
        self.persist_directory = persist_directory
        self.embedding_model = HuggingFaceEmbeddings(model_name="microsoft/harrier-oss-v1-270m")
        self.text_splitter = RecursiveCharacterTextSplitter(chunk_size=600, chunk_overlap=50)
        self.vector_db = None

    def _get_policy_documents(self) -> List[Document]:
        """MIMIC 전용 데이터 변환 및 비식별화 정책 (정적 가이드)"""
        policies = [
            "Policy [Hallucination Prevention]: MIMIC-IV data does not contain real names, addresses, or telecom details. NEVER invent this data (e.g., do not use 'John Smith' or '123 Main St'). Omit these fields or use the FHIR 'data-absent-reason' extension.",
            "Policy [Identifiers]: For the FHIR resource 'id', strictly use the MIMIC 'subject_id' or 'hadm_id' exactly as provided. Do not generate random IDs like 'patient-123'.",
            "Policy [Date Shift]: Dates in MIMIC are shifted into the future for de-identification (e.g., 2180-01-01). Keep the provided dates exactly as they are in 'YYYY-MM-DD' format. Do not calculate or adjust to the current year.",
            "Policy [Local Coding]: Do NOT attempt to map MIMIC 'itemid' to external standards like LOINC or SNOMED-CT. Preserve the local code. Set 'system' to 'http://mimic.mit.edu/fhir/CodeSystem/local-code', 'code' to the 'itemid', and 'display' to the 'label'.",
            "Value Set Mapping [Gender]: Map MIMIC gender 'M' to FHIR 'male' and 'F' to FHIR 'female'.",
            "Value Set Mapping [Encounter Class]: Map MIMIC admission_type 'EMERGENCY' to FHIR 'EMER', 'ELECTIVE' to 'AMB', 'URGENT' to 'URG', and 'OBSERVATION ADMIT' to 'OBSENC'."
        ]
        return [Document(page_content=text, metadata={"category": "policy"}) for text in policies]

    def _get_dictionary_documents(self, gateway: Any) -> List[Document]:
        """MIMIC의 세 가지 주요 사전 테이블에서 로컬 코드를 추출하여 언어화"""
        docs = []
        # 1. d_labitems 처리 (검사 항목)
        print("[*] Extracting Local Codes from d_labitems...")
        lab_items = gateway.execute_safe_query("SELECT itemid, label, fluid, category FROM d_labitems")
        for item in lab_items:
            if 'error' in item: continue
            
            content = (f"MIMIC Local Dictionary [Lab Test]: ItemID '{item['itemid']}' corresponds to the label '{item['label']}'. "
                        f"Category: '{item.get('category', 'N/A')}', Fluid: '{item.get('fluid', 'N/A')}'.")
            docs.append(Document(page_content=content, metadata={"category": "dictionary", "source": "d_labitems", "code": int(item['itemid'])}))

        # 2. d_items 처리 (차트 이벤트, 활력 징후, 처치 등)
        print("[*] Extracting Local Codes from d_items...")
        # d_items는 컬럼이 다르므로 param_type 등을 가져옵니다.
        d_items = gateway.execute_safe_query("SELECT itemid, label, category, param_type FROM d_items")
        for item in d_items:
            if 'error' in item: continue
            
            content = (f"MIMIC Local Dictionary [Chart Item]: ItemID '{item['itemid']}' corresponds to the label '{item['label']}'. "
                        f"Category: '{item.get('category', 'N/A')}', Type: '{item.get('param_type', 'N/A')}'.")
            docs.append(Document(page_content=content, metadata={"category": "dictionary", "source": "d_items", "code": int(item['itemid'])}))

        # 3. d_icd_diagnoses 처리 (진단명)
        print("[*] Extracting Local Codes from d_icd_diagnoses...")
        # 진단 테이블은 itemid 대신 icd_code와 버전을 사용합니다.
        icd_items = gateway.execute_safe_query("SELECT icd_code, icd_version, long_title FROM d_icd_diagnoses")
        for item in icd_items:
            if 'error' in item: continue
            
            content = (f"MIMIC Local Dictionary [Diagnosis]: ICD-{item['icd_version']} code '{str(item.get('icd_code', '')).strip()}' "
                        f"represents the diagnosis '{item['long_title']}'.")
            docs.append(Document(page_content=content, metadata={"category": "dictionary", "source": "d_icd_diagnoses", "code": str(item.get('icd_code', '')).strip()}))

        print(f"[*] 총 {len(docs)}개의 사전 정의 문서가 추출되었습니다.")
        return docs

    def get_or_build_db(self, gateway: Any = None) -> Chroma:
        """Vector DB 로드 또는 새로 구축"""
        # 디렉토리가 존재하고 내부에 파일이 있는지 확인
        if os.path.exists(self.persist_directory) and os.listdir(self.persist_directory):
            print(f"[*] 기존 Medical Knowledge DB 로딩 중: {self.persist_directory}")
            self.vector_db = Chroma(persist_directory=self.persist_directory, embedding_function=self.embedding_model)
        else:
            print(f"[*] Medical Knowledge DB 구축 중 (MIMIC Policies & Local Codes)...")
            all_docs = self._get_policy_documents()
            
            if gateway:
                all_docs.extend(self._get_dictionary_documents(gateway))
            else:
                print("[!] Warning: Gateway 미제공. 사전 데이터 없이 정책 가이드만 구축합니다.")
            split_docs = self.text_splitter.split_documents(all_docs)
            try:
                self.vector_db = Chroma.from_documents(
                    documents=split_docs, 
                    embedding=self.embedding_model, 
                    persist_directory=self.persist_directory
                )
                print(f"[✅] Medical Knowledge DB 구축 완료")
            except Exception as e:
                return split_docs
        return self.vector_db

    def search_context(self, query: str, k: int = 10) -> str:
        """유사도 점수 기반 컨텍스트 반환"""
        if not self.vector_db:
            return "Knowledge base not initialized."
        
        scored_docs = self.vector_db.similarity_search_with_score(query, k=k)
        # L2 Distance 1.5 이하의 유효한 결과만 필터링
        valid_contexts = [doc.page_content for doc, score in scored_docs if score < 1.8]
        
        return "\n\n".join(valid_contexts) if valid_contexts else "No specific clinical or policy evidence found."

# =============================================================================
# 3. 다중 에이전트 파이프라인 (Mapper & Critic)
# =============================================================================

class FHIRMapperOutput(BaseModel):
    thought_process: str = Field(description="Internal reasoning and mapping logic (Chain of Thought)")
    fhir_resource_type: str = Field(description="The target FHIR Resource Type")
    fhir_json: Dict[str, Any] = Field(description="The generated FHIR JSON resource")

class FHIRCriticOutput(BaseModel):
    is_valid: bool = Field(description="Whether the JSON follows FHIR R4 standards")
    error_category: Optional[str] = Field(default=None, description="Category of the error if invalid")
    feedback: str = Field(description="Detailed feedback for the Mapper to fix errors")

class FHIRMultiAgentPipeline:
    def __init__(self, mapper_llm: ChatOllama, critic_llm: ChatOllama, schema_db: Chroma, medical_retriever: MedicalKnowledgeRetriever):
        self.mapper_llm = mapper_llm
        self.critic_llm = critic_llm
        self.schema_db = schema_db
        self.medical_retriever = medical_retriever

        # PROMPT 1: Mapper
        self.mapper_prompt = ChatPromptTemplate.from_template("""
        You are an expert HL7 FHIR Data Mapper. 
        Your task is to map relational database records to a FHIR R4 {target_resource} resource.
        
        [FHIR Schema Knowledge (RAG)]
        {schema_context}
        
        [Medical & Terminology Knowledge (RAG)]
        {medical_context}
        
        [Input Data]
        Source Table: {table_name}
        Raw Data Record: {raw_data}
        
        [Previous Feedback from Critic] 
        {previous_feedback}
        
        Instructions:
        1. Analyze the raw data and schema rules.
        2. Use the RAG context to ensure correct field names, data types, and value sets (enums).
        3. Explain your logic in 'thought_process'.
        4. Output a strict JSON resource.
        
        Response must strictly follow the FHIRMapperOutput schema.
        """)
        self.mapper_chain = self.mapper_prompt | self.mapper_llm.with_structured_output(FHIRMapperOutput)
        
        # PROMPT 2: Critic
        self.critic_prompt = ChatPromptTemplate.from_template("""
        You are a strict HL7 FHIR Verifier/Critic. 
        Evaluate if the provided JSON resource strictly follows FHIR R4 specifications AND matches the Original Raw Data without any hallucinations.
        
        [Original Raw Data]
        {raw_data}
        
        [FHIR Validation Reference (Schema RAG)]
        {schema_context}
        
        [Target JSON for Review]
        {fhir_json}
        
        Validation Checklist:
        1. Are all mandatory fields present?
        2. Are data types strictly correct? (e.g., Encounter.class MUST be a Coding object, not a simple string).
        3. HALLUCINATION CHECK: Are there any values (e.g., names like 'Jane Smith', addresses, telecom, or fake IDs like '123') that DO NOT exist in the Original Raw Data? If yes, it is INVALID.
        
        CRITICAL INSTRUCTION:
        If invalid, your 'feedback' MUST be extremely specific. Tell the Mapper exactly which field failed and why (e.g., "Patient.name is hallucinated, remove it", "Encounter.class is a string but should be a Coding"). DO NOT give generic feedback.
        
        Response must strictly follow the FHIRCriticOutput schema.
        """)
        self.critic_chain = self.critic_prompt | self.critic_llm.with_structured_output(FHIRCriticOutput)

    def run_task(self, task: Dict[str, Any], max_retries: int = 3) -> Optional[Dict[str, Any]]:
        """단일 변환 작업을 수행하며 자가 수정 루프를 실행"""
        target = task['target_resource']
        
        # Dual RAG를 사용하여 관련 컨텍스트 검색
        schema_docs = self.schema_db.similarity_search(f"Resource {target}", k=5)
        schema_ctx = "\n\n".join([d.page_content for d in schema_docs])

        search_query = f"Conversion Policy, Rules, and Dictionary for {target}. Data: {json.dumps(task['data'], default=str)}"
        medical_ctx = self.medical_retriever.search_context(search_query)

        previous_feedback = "This is the initial attempt."
        final_resource = None
        
        print(f"\n[*] 변환 처리 중: {task['table_name']} -> {target}")
        
        for attempt in range(1, max_retries + 1):
            print(f"  ↳ 시도 {attempt}/{max_retries}...")
            try:
                # 1. Mapper 단계
                mapped = self.mapper_chain.invoke({
                    "target_resource": target,
                    "table_name": task['table_name'],
                    "raw_data": json.dumps(task['data'], default=str),
                    "schema_context": schema_ctx,
                    "medical_context": medical_ctx,
                    "previous_feedback": previous_feedback
                })
                
                print(f"    [Mapper] CoT: {mapped.thought_process[:100]}...")
            
                # [디버깅용 추가] Mapper가 만든 JSON을 직접 출력해서 확인
                print(f"    [Mapper] 생성된 JSON:\n{json.dumps(mapped.fhir_json, indent=2)}")
                # 2. Critic 단계
                verified = self.critic_chain.invoke({
                    "target_resource": target,
                    "raw_data": json.dumps(task['data'], default=str),
                    "schema_context": schema_ctx,
                    "fhir_json": json.dumps(mapped.fhir_json, default=str)
                })
                
                # 3. 결정 단계
                if verified.is_valid:
                    print("    [Critic] ✓ 검증 통과.")
                    final_resource = mapped.fhir_json
                    is_success = True
                    break
                else:
                    print(f"    [Critic] ✗ 검증 실패: {verified.feedback}")
                    previous_feedback = verified.feedback
                    
            except Exception as e:
                print(f"    [Error] 시스템 에러 발생: {e}")
                previous_feedback = f"System Error: {str(e)}. Please retry the mapping."
        
        return final_resource

# =============================================================================
# 4. Orchestrator (Manager)
# =============================================================================

def orchestrate_fhir_conversion(patient_id: int):
    """특정 환자에 대한 전체 변환 파이프라인 조율"""
    print(f"\n=== 환자 {patient_id}에 대한 FHIR 변환 오케스트레이션 시작 ===")
    
    # 1. 인프라 초기화
    db_uri = f"postgresql+psycopg2://{config.get('TEST_USER')}:{config.get('TEST_PW')}@{config.get('TEST_HOST')}:{config.get('PORT')}/{config.get('TEST_DB')}"
    gateway = DatabaseGateway(
        uri=db_uri, 
        schemas=["mimiciv_hosp", "mimiciv_icu"], 
        allowed_tables=["patients", "admissions", "d_labitems", "d_items", "d_icd_diagnoses",
        "labevents","chartevents", "diagnoses_icd"]
    )
    
    schema_builder = FHIRSchemaKnowledgeBuilder()
    schema_db = schema_builder.get_or_build_db("./db/fhir.schema.json")
    medical_retriever = MedicalKnowledgeRetriever(persist_directory="./mimic_docs_vectordb")
    
    mapper_llm = ChatOllama(model='qwen3.5:9b', base_url="http://localhost:11434", temperature=0)
    critic_llm = ChatOllama(model='qwen3.5:0.8b', base_url="http://localhost:11434", temperature=0)

    pipeline = FHIRMultiAgentPipeline(
        mapper_llm=mapper_llm, 
        critic_llm=critic_llm, 
        schema_db=schema_db, 
        medical_retriever=medical_retriever
    )
    
    # 2. 메타데이터 및 데이터 조회
    hosp_meta = gateway.get_schema_metadata().get("mimiciv_hosp", {})
    patient_query = f"SELECT * FROM patients WHERE subject_id = {patient_id}"
    p_data_list = gateway.execute_safe_query(patient_query)
    
    if not p_data_list:
        print(f"[!] 환자 ID {patient_id}에 대한 데이터를 찾을 수 없습니다.")
        return

    # 3. 태스크 생성
    tasks = []
    # Patient 리소스 작업 추가
    tasks.append({
        "table_name": "patients",
        "target_resource": "Patient",
        "schema": hosp_meta.get("patients"),
        "data": p_data_list[0]
    })
    
    # Admission 리소스 작업 추가
    adm_query = f"SELECT * FROM admissions WHERE subject_id = {patient_id} LIMIT 1"
    a_data_list = gateway.execute_safe_query(adm_query)
    if a_data_list:
        tasks.append({
            "table_name": "admissions",
            "target_resource": "Encounter",
            "schema": hosp_meta.get("admissions"),
            "data": a_data_list[0]
        })

    # ICU 스키마 메타데이터 추출 (chartevents 용도)
    icu_meta = gateway.get_schema_metadata().get("mimiciv_icu", {})

    # Lab Events (진단검사) -> Observation 리소스
    lab_query = f"SELECT * FROM labevents WHERE subject_id = {patient_id} and hadm_id = {a_data_list[0]['hadm_id']} LIMIT 3"
    lab_data_list = gateway.execute_safe_query(lab_query)
    
    if lab_data_list and 'error' not in lab_data_list[0]:
        print(f"[*] {len(lab_data_list)}개의 Lab Event를 Task로 추가합니다.")
        for lab_data in lab_data_list:
            tasks.append({
                "table_name": "labevents",
                "target_resource": "Observation",
                "schema": hosp_meta.get("labevents"),
                "data": lab_data
            })

    # Chart Events (활력징후 등) -> Observation 리소스
    chart_query = f"SELECT * FROM chartevents WHERE subject_id = {patient_id} and hadm_id = {a_data_list[0]['hadm_id']} LIMIT 3"
    chart_data_list = gateway.execute_safe_query(chart_query)
    
    if chart_data_list and 'error' not in chart_data_list[0]:
        print(f"[*] {len(chart_data_list)}개의 Chart Event를 Task로 추가합니다.")
        for chart_data in chart_data_list:
            tasks.append({
                "table_name": "chartevents",
                "target_resource": "Observation",
                "schema": icu_meta.get("chartevents"),
                "data": chart_data
            })

    # Diagnoses ICD (진단 내역) -> Condition 리소스
    diag_query = f"SELECT * FROM diagnoses_icd WHERE subject_id = {patient_id} LIMIT 3"
    diag_data_list = gateway.execute_safe_query(diag_query)
    
    if diag_data_list and 'error' not in diag_data_list[0]:
        print(f"[*] {len(diag_data_list)}개의 Diagnosis를 Task로 추가합니다.")
        for diag_data in diag_data_list:
            tasks.append({
                "table_name": "diagnoses_icd",
                "target_resource": "Condition",
                "schema": hosp_meta.get("diagnoses_icd"),
                "data": diag_data
            })

    # 4. 파이프라인 실행
    fhir_bundle = {
        "resourceType": "Bundle",
        "type": "transaction",
        "entry": []
    }
    # 환자 전용 출력 폴더 생성
    output_dir = f"./fhir_output/patient_{patient_id}"
    os.makedirs(output_dir, exist_ok=True)
    print(f"\n[*] 결과물 저장 폴더 생성: {output_dir}")

    saved_files = []
    for t in tasks:
        resource = pipeline.run_task(t)
        
        if resource:
            # 1. Bundle에 추가
            fhir_bundle["entry"].append({
                "resource": resource,
                "request": {"method": "POST", "url": t['target_resource']}
            })
            
            # 2. 개별 리소스 파일 생성 로직
            # 생성된 JSON에서 resourceType과 id를 추출하여 파일명 지정
            res_type = resource.get("resourceType", t['target_resource'])
            # 만약 id 생성을 실패했을 경우 테이블 이름으로 대체 (안전 장치)
            res_id = resource.get("id", f"unknown_{t['table_name']}") 
            
            individual_filename = os.path.join(output_dir, f"{res_type}_{res_id}.json")
            
            with open(individual_filename, "w", encoding='utf-8') as f:
                json.dump(resource, f, indent=2, ensure_ascii=False)
                
            saved_files.append(individual_filename)
            print(f"    [Save] 개별 리소스 저장 완료: {individual_filename}")
    
    # 5. Bundle 파일도 별도로 저장 (옵션)
    bundle_filename = os.path.join(output_dir, f"Bundle_transaction.json")
    with open(bundle_filename, "w", encoding='utf-8') as f:
        json.dump(fhir_bundle, f, indent=2, ensure_ascii=False)
    print("\n=== 오케스트레이션 완료 ===")
    print(f"생성된 파일 목록 ({len(saved_files)}개 리소스 + 1개 번들):")

# if __name__ == "__main__":
#     # MIMIC-IV 샘플 Subject ID
#     TEST_SUBJECT_ID = '10000032'
#     orchestrate_fhir_conversion(TEST_SUBJECT_ID)

In [ ]:
x

NameError: name 'x' is not defined

In [ ]:
from pydantic import BaseModel, Field, ValidationError, validator
from datetime import date, datetime
from typing import Optional, Any, Dict
from config import config

# --- [추가] DB 입력 데이터 검증용 모델 ---

class PatientInput(BaseModel):
    subject_id: int
    gender: str = Field(..., pattern="^[MF]$") # M 또는 F만 허용
    anchor_age: int = Field(..., ge=0, le=120)
    anchor_year: int
    dod: Optional[Any] = None # date 객체 또는 None

class EncounterInput(BaseModel):
    subject_id: int
    hadm_id: int
    admission_type: str
    admittime: datetime
    dischtime: Optional[datetime] = None

class ObservationInput(BaseModel):
    subject_id: int
    itemid: int
    charttime: datetime
    valuenum: Optional[float] = None
    valueuom: Optional[str] = None

class ConditionInput(BaseModel):
    subject_id: int
    hadm_id: int
    icd_code: str
    icd_version: int

def validate_input_data(table_name: str, data: Dict[str, Any]) -> Optional[Dict[str, Any]]:
    """DB에서 추출된 로우 데이터가 최소 규격에 맞는지 검증"""
    models = {
        "patients": PatientInput,
        "admissions": EncounterInput,
        "labevents": ObservationInput,
        "chartevents": ObservationInput,
        "diagnoses_icd": ConditionInput
    }
    
    model = models.get(table_name)
    if not model:
        return data # 모델이 없는 테이블은 그대로 통과
        
    try:
        # Pydantic 모델로 파싱하여 검증 수행
        validated_obj = model(**data)
        return validated_obj.model_dump() # 검증된 딕셔너리 반환
    except ValidationError as e:
        print(f"    [!] 입력 데이터 검증 실패 ({table_name}): {e.json()}")
        return None

In [ ]:
from pydantic import ValidationError
from fhir.resources.patient import Patient
from fhir.resources.observation import Observation
from fhir.resources.encounter import Encounter
from fhir.resources.condition import Condition

def validate_fhir_standard(resource_type: str, fhir_json: dict) -> Tuple[bool, str]:
    """
    공식 FHIR 라이브러리를 사용하여 기술적 규격 검증
    """
    resource_classes = {
        "Patient": Patient,
        "Observation": Observation,
        "Encounter": Encounter,
        "Condition": Condition
    }
    
    model_class = resource_classes.get(resource_type)
    if not model_class:
        return True, "" # 지원하지 않는 리소스는 통과

    try:
        # 공식 규격에 맞는지 파싱 시도
        model_class.parse_obj(fhir_json)
        return True, ""
    except Exception as e:
        # 구체적인 에러 메시지를 Critic에게 넘겨줄 피드백으로 활용
        return False, str(e)

def get_dynamic_patient_task(llm, gateway, schema_db, target_resource, patient_id, max_retries=3):
    """
    LLM이 쿼리를 생성하고, 실패 시 피드백을 받아 스스로 수정(Self-Correction)하는 함수
    """
    db_metadata = gateway.get_schema_metadata()
    
    # FHIR 표준 정의 가져오기
    fhir_schema_docs = schema_db.similarity_search(f"Resource definition and properties of {target_resource}", k=10)
    fhir_schema_context = "\n".join([d.page_content for d in fhir_schema_docs])
    
    previous_error = "None" # 이전 시도의 에러 메시지 저장용
    
    for attempt in range(1, max_retries + 1):
        print(f"[*] {target_resource} 매핑 시도 {attempt}/{max_retries}...")
        
        explorer_prompt = f"""
        You are a SQL Expert. Map DB Columns to FHIR Fields.
        
        [FHIR Standard]
        {fhir_schema_context}
        
        [DB Metadata]
        {json.dumps(db_metadata, indent=2)}
        
        [Target]
        Resource: {target_resource}, SubjectID: {patient_id}
        
        [Previous Error Feedback]
        {previous_error}
        
        [Instructions]
        1. Generate a PostgreSQL SELECT query.
        2. If 'Previous Error Feedback' is not None, FIX the query to avoid that error.
        3. Ensure all selected columns exist in the DB Metadata.
        4. Return ONLY a JSON object: {{"table_name": "...", "sql_query": "...", "explanation": "..."}}
        """
        
        response = llm.invoke(explorer_prompt)
        print(response.content)
        try:
            # 1. LLM 응답 파싱
            decision = json.loads(response.content)
            table_name = decision['table_name']
            sql_query = decision['sql_query']
            
            # 2. 쿼리 실행 테스트
            raw_results = gateway.execute_safe_query(sql_query)
            
            # 3. 결과 검증 (에러 발생 여부 확인)
            if raw_results and isinstance(raw_results[0], dict) and 'error' in raw_results[0]:
                error_msg = raw_results[0]['error']
                print(f"    [!] 쿼리 실행 실패: {error_msg}")
                previous_error = f"SQL Error: {error_msg}. Please check column names or syntax."
                continue # 다음 리트라이로 진행
            
            if not raw_results:
                print(f"    [!] 데이터가 존재하지 않습니다.")
                previous_error = "Query returned no results. Check if the table or subject_id is correct."
                continue

            # 4. 성공 시 반환
            print(f"    [✓] 매핑 성공: {table_name}")
            return {
                "table_name": table_name,
                "target_resource": target_resource,
                "data": raw_results[0]
            }
            
        except Exception as e:
            print(f"    [!] 시도 {attempt} 중 시스템 오류: {e}")
            previous_error = f"Processing Error: {str(e)}"
            
    print(f"    [✘] {max_retries}회 시도 모두 실패했습니다.")
    return None

# 1. 인프라 초기화
db_uri = f"postgresql+psycopg2://{config.get('TEST_USER')}:{config.get('TEST_PW')}@{config.get('TEST_HOST')}:{config.get('PORT')}/{config.get('TEST_DB')}"
gateway = DatabaseGateway(
    uri=db_uri, 
    schemas=["mimiciv_hosp", "mimiciv_icu"], 
    allowed_tables=["patients", "admissions", "d_labitems", "d_items", "d_icd_diagnoses",
    "labevents","chartevents", "diagnoses_icd"]
)

llm = ChatOllama(model='qwen3.5:0.8b', base_url="http://localhost:11434", temperature=0)
schema_builder = FHIRSchemaKnowledgeBuilder()
schema_db = schema_builder.get_or_build_db("./db/fhir.schema.json")
result = get_dynamic_patient_task(llm, gateway, schema_db, "Patient", 10000032)

/tmp/ipykernel_732247/3962023133.py:100: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  self.embedding_model = HuggingFaceEmbeddings(model_name="microsoft/harrier-oss-v1-270m")


[*] 기존 FHIR Schema Vector DB 로딩 중: ./fhir_schema_vectordb


/tmp/ipykernel_732247/3962023133.py:135: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  return Chroma(persist_directory=self.persist_directory, embedding_function=self.embedding_model)


[*] Patient 매핑 시도 1/3...
